# Public Benchmark Baseline: CatBoost on Leak-Free Aggregate Features


In [ ]:
# Cell 1 - Setup & Install
import json
import os
import pickle
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import yaml
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyyaml", "-q"])
    import yaml

REPO_DIR = Path("/kaggle/working/denoising-event-sequences")
if not REPO_DIR.exists():
    REPO_DIR = Path.cwd()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DATA_ROOT = Path("/kaggle/working/data")
RAW_ROOT = DATA_ROOT / "raw"
PROCESSED_ROOT = DATA_ROOT / "processed"
OUTPUT_ROOT = Path("/kaggle/working/outputs/public_benchmarks")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run_logged(cmd, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(map(str, cmd)))
    with log_path.open("w") as log_file:
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=str(REPO_DIR),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    print(f"Log saved: {log_path}")

try:
    from catboost import CatBoostClassifier, Pool
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "catboost", "-q"])
    from catboost import CatBoostClassifier, Pool


In [ ]:
# Shared - Prepare Public Benchmark Data If Missing
DATASET_NAME = "gender"  # "gender" or "age_group"
DATASET_CONFIG_PATHS = {
    "gender": REPO_DIR / "configs" / "datasets" / "gender.yaml",
    "age_group": REPO_DIR / "configs" / "datasets" / "age_group.yaml",
}
PROCESSED_DIR = PROCESSED_ROOT / f"{DATASET_NAME}_benchmark"
RUN_PREPARE_DATA_IF_MISSING = True
FORCE_REBUILD_PROCESSED = False
MAX_SEQ_LEN = 512

required_artifacts = ["events.parquet", "canonical_events.parquet", "transformed_events.parquet", "splits.json", "preprocessor.pkl", "prepared_config.yaml", "data_report.json"]
artifacts_exist = all((PROCESSED_DIR / name).exists() for name in required_artifacts)
if artifacts_exist:
    with (PROCESSED_DIR / "data_report.json").open() as f:
        existing_report = json.load(f)
    artifacts_exist = bool(existing_report.get("events_are_raw_pretransform", False))
if FORCE_REBUILD_PROCESSED and PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)
    artifacts_exist = False

if RUN_PREPARE_DATA_IF_MISSING and not artifacts_exist:
    run_logged(
        [
            sys.executable,
            REPO_DIR / "scripts" / "prepare_public_benchmark_data.py",
            "--dataset", DATASET_NAME,
            "--raw-root", RAW_ROOT,
            "--output-dir", PROCESSED_DIR,
            "--max-seq-len", MAX_SEQ_LEN,
        ],
        OUTPUT_ROOT / "logs" / f"{DATASET_NAME}_prepare_public_benchmark_data.log",
    )

assert all((PROCESSED_DIR / name).exists() for name in required_artifacts), PROCESSED_DIR
with (PROCESSED_DIR / "data_report.json").open() as f:
    data_report = json.load(f)
print(json.dumps(data_report, indent=2)[:3000])


In [ ]:
# Shared - Multiclass-Safe Metrics
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize


def classification_metrics(y_true, y_proba, num_classes):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=float)
    y_pred = y_proba.argmax(axis=1)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }
    if num_classes == 2:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_proba[:, 1]))
        metrics["pr_auc"] = float(average_precision_score(y_true, y_proba[:, 1]))
    else:
        y_bin = label_binarize(y_true, classes=list(range(num_classes)))
        metrics["roc_auc_ovr"] = float(roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro"))
        metrics["macro_pr_auc"] = float(average_precision_score(y_bin, y_proba, average="macro"))
    return metrics


In [ ]:
# Cell 4 - Load Canonical Data and Shared Splits
with (PROCESSED_DIR / "prepared_config.yaml").open() as f:
    cfg = yaml.safe_load(f)
canonical_df = pd.read_parquet(PROCESSED_DIR / "canonical_events.parquet")
with (PROCESSED_DIR / "splits.json").open() as f:
    splits = json.load(f)

entity_col = cfg["data"]["group_col"]
time_col = cfg["data"]["timestamp_col"]
target_col = cfg["data"]["target_col"]
event_col = cfg["data"]["event_type_col"]
num_cols = list(cfg["data"].get("numerical_cols") or [])
cat_cols = list(cfg["data"].get("categorical_cols") or [])
num_classes = int(cfg["training"].get("num_classes", 2))

PREFIX_FRAC = 0.70
TOP_K_EVENT_TYPES = 64
TOP_K_CATEGORICAL = 32
SMOKE_RUN = False
CATBOOST_CATEGORICAL_FEATURES = ["first_event", "last_event", "mode_event"] + [f"first_{c}" for c in cat_cols] + [f"last_{c}" for c in cat_cols] + [f"mode_{c}" for c in cat_cols]
CATBOOST_FITTED_FEATURE_METADATA = {}

print(canonical_df.shape, {k: len(v) for k, v in splits.items()}, num_classes)


In [ ]:
# Cell 5 - Prefix-Only Feature Builder

def prefix_only(df):
    pieces = []
    cutoffs = []
    for _, group in df.sort_values([entity_col, time_col], kind="stable").groupby(entity_col, sort=False):
        n = len(group)
        cutoff = max(1, min(n, int(np.ceil(n * PREFIX_FRAC))))
        pref = group.iloc[:cutoff].copy()
        pieces.append(pref)
        cutoffs.append({entity_col: group[entity_col].iloc[0], "prefix_events": cutoff, "full_events": n})
    return pd.concat(pieces, ignore_index=True), pd.DataFrame(cutoffs)


def fit_top_values(train_prefix):
    metadata = {"event_top_values": train_prefix[event_col].astype(str).value_counts().head(TOP_K_EVENT_TYPES).index.tolist(), "cat_top_values": {}}
    for c in cat_cols:
        metadata["cat_top_values"][c] = train_prefix[c].astype(str).value_counts().head(TOP_K_CATEGORICAL).index.tolist()
    return metadata


def recency_weighted_counts(group, col, values):
    vals = group[col].astype(str).to_numpy()
    weights = np.linspace(0.25, 1.0, num=len(vals), dtype=float)
    denom = float(weights.sum()) or 1.0
    return {f"recency_weighted_{col}_{v}": float(weights[vals == v].sum() / denom) for v in values}


def build_feature_frame(prefix_df, metadata):
    rows = []
    event_values = metadata["event_top_values"]
    for entity_id, group in prefix_df.sort_values([entity_col, time_col], kind="stable").groupby(entity_col, sort=False):
        row = {entity_col: entity_id, "event_count": len(group), "target": int(group[target_col].iloc[0])}
        times = group[time_col].astype(float if np.issubdtype(group[time_col].dtype, np.number) else "datetime64[ns]")
        row["span"] = float(pd.Series(times).max() - pd.Series(times).min()) if np.issubdtype(group[time_col].dtype, np.number) else float((times.max() - times.min()).total_seconds())
        row["first_event"] = str(group[event_col].iloc[0])
        row["last_event"] = str(group[event_col].iloc[-1])
        row["mode_event"] = str(group[event_col].astype(str).mode().iloc[0])
        vc = group[event_col].astype(str).value_counts(normalize=True)
        for v in event_values:
            row[f"event_share_{v}"] = float(vc.get(v, 0.0))
        row.update(recency_weighted_counts(group, event_col, event_values))
        for c in cat_cols:
            row[f"first_{c}"] = str(group[c].iloc[0])
            row[f"last_{c}"] = str(group[c].iloc[-1])
            row[f"mode_{c}"] = str(group[c].astype(str).mode().iloc[0])
            vc_c = group[c].astype(str).value_counts(normalize=True)
            for v in metadata["cat_top_values"].get(c, []):
                row[f"{c}_share_{v}"] = float(vc_c.get(v, 0.0))
        for c in num_cols:
            values = pd.to_numeric(group[c], errors="coerce").fillna(0.0)
            row[f"{c}_mean"] = float(values.mean())
            row[f"{c}_std"] = float(values.std(ddof=0))
            row[f"{c}_min"] = float(values.min())
            row[f"{c}_max"] = float(values.max())
            row[f"{c}_sum"] = float(values.sum())
            row[f"{c}_tail3_mean"] = float(values.tail(3).mean())
        rows.append(row)
    features = pd.DataFrame(rows).fillna(0.0)
    return features


def assert_no_future_leakage(prefix_meta):
    assert (prefix_meta["prefix_events"] <= prefix_meta["full_events"]).all()
    assert (prefix_meta["prefix_events"] >= 1).all()


def assert_catboost_feature_frame_is_leak_safe(features, split_ids):
    assert set(features[entity_col]).issubset(set(split_ids))
    forbidden = {target_col, "full_events"}
    assert not (forbidden & set(features.columns))


In [ ]:
# Cell 6 - Train CatBoost
train_full = canonical_df[canonical_df[entity_col].isin(splits["train"])].copy()
train_prefix, train_prefix_meta = prefix_only(train_full)
assert_no_future_leakage(train_prefix_meta)
CATBOOST_FITTED_FEATURE_METADATA = fit_top_values(train_prefix)

feature_frames = {}
for split_name, ids in splits.items():
    split_full = canonical_df[canonical_df[entity_col].isin(ids)].copy()
    split_prefix, split_meta = prefix_only(split_full)
    assert_no_future_leakage(split_meta)
    feature_frames[split_name] = build_feature_frame(split_prefix, CATBOOST_FITTED_FEATURE_METADATA)
    assert_catboost_feature_frame_is_leak_safe(feature_frames[split_name], ids)

train_features = feature_frames["train"]
val_features = feature_frames["val"]
test_features = feature_frames["test"]
feature_cols = [c for c in train_features.columns if c not in {entity_col, "target"}]
cat_feature_names = [c for c in CATBOOST_CATEGORICAL_FEATURES if c in feature_cols]

X_train, y_train = train_features[feature_cols], train_features["target"].astype(int)
X_val, y_val = val_features[feature_cols], val_features["target"].astype(int)
X_test, y_test = test_features[feature_cols], test_features["target"].astype(int)

iterations = 100 if SMOKE_RUN else 3000
model = CatBoostClassifier(
    iterations=iterations,
    learning_rate=0.025,
    depth=6,
    l2_leaf_reg=8.0,
    loss_function="Logloss" if num_classes == 2 else "MultiClass",
    eval_metric="AUC" if num_classes == 2 else "TotalF1",
    auto_class_weights="Balanced",
    bootstrap_type="Bayesian",
    random_seed=42,
    od_type="Iter",
    od_wait=120,
    verbose=100,
)
model.fit(
    Pool(X_train, y_train, cat_features=cat_feature_names),
    eval_set=Pool(X_val, y_val, cat_features=cat_feature_names),
    use_best_model=True,
)

test_proba = model.predict_proba(X_test)
metrics = classification_metrics(y_test, test_proba, num_classes)
metrics


In [ ]:
# Cell 7 - Save Metrics and Feature Importance
out_dir = OUTPUT_ROOT / DATASET_NAME / "catboost_agg"
out_dir.mkdir(parents=True, exist_ok=True)
metrics_path = out_dir / "catboost_agg_metrics.json"
with metrics_path.open("w") as f:
    json.dump({"metrics": metrics, "feature_metadata": CATBOOST_FITTED_FEATURE_METADATA}, f, indent=2)
importance_df = pd.DataFrame({"feature": feature_cols, "importance": model.get_feature_importance()}).sort_values("importance", ascending=False)
importance_df.to_csv(out_dir / "catboost_agg_feature_importance.csv", index=False)
print(metrics_path)
importance_df.head(30)
